In [1]:
from PIL import Image
import sys
sys.path.insert(0, "../")
from src.model.gigachat_vl import GigaChatVL, GigaChatVLForInference, IGNORE_INDEX
from src.dataset.finevision import VLMDataCollator
from src.utils.train_utils import save_artifacts
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch

In [2]:
LLM_PATH = "/media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16"
VISION_PATH = "/media/alexey/HDDLargeData/models/VLM/Qwen2.5-VL-7B-Instruct/"

sample1 = {
    "image": "/home/alexey/Downloads/tg.jpg",
    "question": "Извлеки текст из изображения",
    "answer": "Document Liberation Own your content",
}

sample2 = {
    "image": "/home/alexey/Downloads/tg2.jpg",
    "question": "Извлеки текст из изображения",
    "answer": "Привет, что делаешь? Вид, что всё хорошо.",
}

In [3]:
class TwoSampleDataset(Dataset):
    def __init__(self, repeats=200):
        self.samples = [sample1, sample2] * repeats

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

model = GigaChatVL(
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=True,
    freeze_vision=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
)

collator = VLMDataCollator(model=model, max_length=256)
train_dataset = TwoSampleDataset(repeats=200)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args = TrainingArguments(
    output_dir="/tmp/gigachat_vl_two_image_test",
    max_steps=120,
    learning_rate=5e-4,
    weight_decay=0.0,
    warmup_steps=0,
    lr_scheduler_type="constant",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=10,
    save_strategy="no",
    bf16=use_bf16,
    fp16=use_fp16,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
)

trainer.train()
save_artifacts(model, "/tmp/gigachat_vl_two_image_export")

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: /media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.down_proj.weight | UNEXPECTED |  | 
model.layers.26.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.26.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.26.self_attn.q_proj.weight             | UNEXPECTED |  | 
model.layers.26.embed_tokens.weight                 | UNEXPECTED |  | 
model.layers.26.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.26.enorm.weight                        | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.26.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_pro

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Step,Training Loss
10,4.945685
20,1.410093
30,0.464794
40,0.084312
50,0.003785
60,0.000088
70,0.000039
80,0.000017
90,0.000014
100,0.000009


/home/alexey/venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


In [3]:
infer_model = GigaChatVLForInference(
    checkpoint_dir="/tmp/gigachat_vl_two_image_export",
    llm_name=LLM_PATH,
    vision_name=VISION_PATH,
    use_4bit_llm=True,
)

img1 = Image.open("/home/alexey/Downloads/tg.jpg").convert("RGB")
img2 = Image.open("/home/alexey/Downloads/tg2.jpg").convert("RGB")
blank = Image.new("RGB", img1.size, "white")

prompt = "Извлеки текст из изображения"

print("IMG1:")
print(infer_model.inference(prompt, image=img1, do_sample=False, max_new_tokens=48))
print()

print("IMG2:")
print(infer_model.inference(prompt, image=img2, do_sample=False, max_new_tokens=48))
print()

print("BLANK:")
print(infer_model.inference(prompt, image=blank, do_sample=False, max_new_tokens=48))
print()

print("NONE:")
print(infer_model.inference(prompt, image=None, do_sample=False, max_new_tokens=48))

Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

DeepseekV3ForCausalLM LOAD REPORT from: /media/alexey/HDDLargeData/models/VLM/GigaChat3.1-10B-A1.8B-bf16
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.26.shared_head.norm.weight             | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.down_proj.weight | UNEXPECTED |  | 
model.layers.26.eh_proj.weight                      | UNEXPECTED |  | 
model.layers.26.mlp.experts.down_proj               | UNEXPECTED |  | 
model.layers.26.self_attn.q_proj.weight             | UNEXPECTED |  | 
model.layers.26.hnorm.weight                        | UNEXPECTED |  | 
model.layers.26.post_attention_layernorm.weight     | UNEXPECTED |  | 
model.layers.26.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.26.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.26.embed_tokens.weight                 | UNEXPECTED |  | 
model.layers.26.mlp.shared_experts.gate_pro

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

IMG1:
Document Liberation Own your content

IMG2:
Привет, что делаешь? Вид, что всё хорошо.

BLANK:
Document Liberation Own your content

NONE:
Document Liberation Own your content
